# 和集合（アンサンブル）を測る

**この雛形は `docs/ノートブックの作り方.md` の `5` の手順そのものである。**

- **構成員は `ensembles/*.yaml` にだけ書く。** ここに名前を並べない
- **成否は箇所F ではなく「床」で見る**（どの構成員も1画素も光らない
  正解の割合）。和集合だけでは箇所F は上がらない（`42.1`）
- **数字はスクリプトに出させる。** `42`〜`44` を出した
  `analysis/run_union_blind.py` と `analysis/run_union_subsets.py` を
  そのまま呼ぶので、ここで別に計算して食い違うことがない

In [ ]:
import subprocess
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "dc5lib" / "paths.py").exists():
        ROOT = _p
        sys.path.insert(0, str(_p))
        break

from dc5lib import ensemble

ENSEMBLE = "union5"          # ensembles/<これ>.yaml
REGIONS = ["hiroshima", "shimane"]

e = ensemble.get(ENSEMBLE)
print(ensemble.report(e))

In [ ]:
# ============================================================
# **混ぜてよいかを確かめる。** だめなら例外で止まる。
#   ・構成員が実在し、重みがあるか
#   ・ensemble: false が入っていないか（分割か選択が他と違う条件）
#   ・重みの素性から分割が一致するか
# ============================================================
v = ensemble.verify(e)        # 問題があれば AssertionError
print("検査を通りました")
if v["unknown"]:
    n = len(v["unknown"])
    print(f"※ 素性の無い重みが {n} 本あるので、分割は次のセルで pkl から確かめる")

In [ ]:
# ============================================================
# 分割が全員一致するかを pkl から確かめる。**素性の無い古い重みが
# 1本でもあれば必ず通すこと。** ここを飛ばすと、構成員Aの学習タイルが
# Bのテストタイルに入っていても気づけない（和集合の再現率が膨らむ）。
# pkl を読むので広島+島根で5分ほどかかる。
# ============================================================
if v["unknown"]:
    for region in REGIONS:
        r = subprocess.run(
            [sys.executable, "tools/audit_splits.py", "--region", region,
             "--conditions", *e.all],
            cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
        print(r.stdout[-2500:])
        assert "食い違った組 0" in r.stdout, f"{region} で分割が食い違っている"
else:
    print("重みの素性で確かめ済み（pkl を読む必要なし）")

In [ ]:
# ============================================================
# 本体。単体と和集合の箇所評価・床・天井・面積層別・合意数を出す。
# **42〜44 の数字を出したスクリプトそのもの。**
# 構成員ぶんの推論が走るので、広島で数分・島根で数十分かかる。
# ============================================================
for region in REGIONS:
    r = subprocess.run(
        [sys.executable, "analysis/run_union_blind.py",
         "--region", region, "--save", "--members", *e.all],
        cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit(f"{region} で失敗しました")

In [ ]:
# ============================================================
# 部分集合ごとの床と、光らせる先の重なり（Jaccard）。
# **推論しない。** 上で保存した CSV を読むだけ。
# 1本足したときに床がどれだけ下がったか、場所が重なっているかを見る。
# ============================================================
base = [n.replace("FinalFusion_", "").replace("_OverlapTrain", "")
        for n in [e.base] + e.members[:2]]
cands = [n.replace("FinalFusion_", "").replace("_OverlapTrain", "")
         for n in e.members[2:]] or None
cmd = [sys.executable, "analysis/run_union_subsets.py",
       "--region", "both", "--base", *base]
if cands:
    cmd += ["--candidates", *cands]
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True,
                   encoding="utf-8")
print(r.stdout)

## 読み方

| 見るもの | 意味 |
|---|---|
| **床**（`【5】` の光った本数 0 の行）| どの構成員も1画素も光らない正解。**ここが下がったかで成否を決める** |
| 天井（1 − 床）| 構成員を増やして届きうる上限 |
| 刈り取り後の上限 箇所F | `2R/(1+R)` ── 完璧に刈り取れたときの値 |
| 和集合の箇所F | **上がらないのが普通。** 価値は上限にある（`42.1`） |
| Jaccard | 1 に近いほど同じ場所を光らせている。低いほど手札として重ならない |

**ゆらぎ幅**（広島 箇所F 0.0071 / 島根 0.0198）より小さい差は「差が無い」。

## 次にやること

1. 床が下がったら `ensembles/` の定義に `recorded:` を書き、docs に数字を残す
2. 刈り取りまで通す → `python analysis/run_stage2_eval.py --union-with ...`
3. **新しい構成員を足すときは、先に分割の一致を確かめる**
   （上のセル。`ensemble: false` の条件は混ぜない）